In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Q1. How many lesson pages

In [3]:
len(documents)

72

# Q2. Indexing and searching

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query, num_results=5)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [5]:
for r in results:
    print(r["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


# Q3. RAG

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase

load_dotenv()

class LessonRAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        return "\n\n".join(
            f"{doc['filename']}\n{doc['content']}" for doc in search_results
        )

    def llm(self, prompt):
        return self.llm_client.responses.create(
            model=self.model,
            input=[
                {"role": "developer", "content": self.instructions},
                {"role": "user", "content": prompt},
            ],
        )

    def rag(self, query):
        response = self.llm(self.build_prompt(query, self.search(query)))
        return response.output_text, response.usage

rag = LessonRAG(index=index, llm_client=OpenAI(), model="gpt-5.4-mini")
answer, usage = rag.rag(query)
usage.input_tokens

7111

# Q4. Chunking

In [8]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

# Q5. RAG with chunking

In [10]:
from gitsource import chunk_documents
from minsearch import Index
client = OpenAI()
chunks = chunk_documents(documents, size=2000, step=1000)
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)
rag = LessonRAG(index=chunk_index, llm_client=client, model="gpt-5.4-mini")
_, usage_chunks = rag.rag(query)
usage_chunks.input_tokens

2294

# Q6. Turning it into an agent

In [11]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> list[dict]:
    """Search course lesson chunks for entries matching the query."""
    return chunk_index.search(query, num_results=5)

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

# count search tool calls
search_calls = sum(
    1 for msg in result.all_messages
    if getattr(msg, "name", None) == "search"
)
search_calls

-> Response received


-> Response received


4